# 70 — Train BGE-M3 bi-encoder (Stage A of nDCG-stretch plan)

Fine-tunes BAAI/bge-m3 on per-music-turn conversation pairs walked
from `talkpl-ai/TalkPlayData-Challenge-Dataset` (train split) via the
Task 6 builder. Uses a custom PEFT-LoRA training loop
(sentence-transformers + peft) — NOT FlagEmbedding's CLI, which
lacks the LoRA flags this plan needs.

**Prereqs**: HF_TOKEN in Colab Secrets. Drive folder
`recsys2026_retrieval_v2_cache` exists.

**Wallclock**: ~8-12 hr on Blackwell (1.5 hr HN mining + 6-10 hr train).

In [ ]:
# 1) Setup — clone + HF auth + Drive mount + deps.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q --upgrade \
    'peft>=0.10' 'transformers>=4.40' 'accelerate>=0.30' \
    'sentence-transformers>=3.0' 'FlagEmbedding>=1.3' \
    'datasets' 'pandas<3.0' 'tqdm' 'omegaconf' 'pyyaml' 'tensorboard'

In [ ]:
# 2) Smoke: build 200 triples to verify the HN miner works end-to-end.
# IMPORTANT: --train-conv-hf walks the HF conversation dataset directly
# (Task 6's `_iter_conversation_turns`). Do NOT pass --train-parquet:
# the W2 parquet schema is (source, session_id, track_id, query,
# code_1..3) and lacks chat_history / current_user_query /
# user_profile_raw / conversation_goal that the builder needs.
!cd /content/recsys2026 && python scripts/build_bi_encoder_training_data.py \
    --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
    --output experiments/cache/retrieval_v2/triples_smoke.jsonl \
    --max-rows 200 --percpos-threshold 0.80 --k-negs 15 \
    2>&1 | tail -20
!wc -l experiments/cache/retrieval_v2/triples_smoke.jsonl
!head -3 experiments/cache/retrieval_v2/triples_smoke.jsonl

In [ ]:
# 3) Full HN mining — ~1.5 hr on Blackwell.
import os
RESULTS_DIR = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
!cd /content/recsys2026 && python -u scripts/build_bi_encoder_training_data.py \
    --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
    --output experiments/cache/retrieval_v2/triples_bge_m3.jsonl \
    --percpos-threshold 0.80 --k-negs 15 --batch-size 64 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/hn_mining_log.txt
!wc -l experiments/cache/retrieval_v2/triples_bge_m3.jsonl